REDES NEURONALES - OPTIMIZACION

Cargamos las librerias que vamos a usar:
torch la base que proporciona las funcionalidades necesarias para construir y entrenar redes neuronales. 
torchvision es una libreria especializada en vision por computadora, la utilizamos para simplificar la carga y manipulaacion de datos de imagenes. Ofrece: transformacion de datos(Resize, crop, convertir a tensor, Image folder que basandose en la estructura de carpetas asignara categorias).

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

Preparamos las imagenes utilizando transformaciones para redimensionar y normalizar las imagenes. transforms.Compose es un clase de torchvision que permite encadenar varias transformaciones en un solo objeto. transform.Normalize normaliza los valores de los pixeles de cada color (Rojo, Verde, Azul)

In [2]:
data_transforms = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

Cargamos los datasets.

In [3]:
train_data_path = "C:/Users/asus/Documents/SIS420-Inteligencia-Artificial-I-/Laboratorios/Laboratorio6/train"
val_data_path = 'C:/Users/asus/Documents/SIS420-Inteligencia-Artificial-I-/Laboratorios/Laboratorio6/val'

In [4]:
train_dataset = torchvision.datasets.ImageFolder(root=train_data_path, transform=data_transforms)
val_dataset = torchvision.datasets.ImageFolder(root=val_data_path, transform=data_transforms)

Creamos DataLoaders que nos permitiran iterar sobre los datos en lotes (batches), es decir ayuda a cargar un pequeño grupo de imagenes (un lote) a la vez, las pasa a la GPU para el entrenamiento y luego carga el siguiente lote.  

In [5]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [6]:
dataloader = {
    'train': train_dataloader,
    'val': val_dataloader
}

Verificamos la estructura de nuestros datos:

In [7]:
torch.cuda.is_available()

True

In [8]:
print(f'Clases encontradas: {train_dataset.classes}')
print(f'Número de imágenes de entrenamiento: {len(train_dataset)}')
print(f'Número de imágenes de validación: {len(val_dataset)}')

Clases encontradas: ['cat', 'dog', 'elephant', 'horse', 'lion']
Número de imágenes de entrenamiento: 13474
Número de imágenes de validación: 1497


Cuando se trabaja con imagenes en una red neuronalm, los datos no ingresan como archivos .jpg o .png (estamos trabajando con imagenes). En su lugar, se convierten en tensores(Matrices numericas que la red puede procesar).
torchvision se encarga de cargar la imagen, y una vez leida la imagen se transforma en un tensor. Cada pixel de la imagen se convierte en un numero. Para una imagen a color (RGB) se crean 3 matrices apiladas (una para el canal rojo, otra para el  verde y una para el azul). Los valores de los pixeles, que suelen estar en un rango de 0 a 255, se escalan a un rango de 0.0 a 1.0. Esto hace que los calculos sean mas eficientes y evita que algunos canales de color dominen el proceso dde aprendizaje.
Despues de una normalizacion y redimensionamiento, el tenssor final tiene una forma tridimensional [canales, altura, anchura]. Dimension de un tensor  32(numero de batch) *3*32*32 = 393.216

Verificamos un Lote:

In [9]:
for inputs, labels in train_dataloader:
    # La forma del tensor 'inputs' es (batch_size, channels, height, width)
    print(f"Número de lotes (batch size): {inputs.shape[0]}")
    print(f"Número de canales (RGB/escala de grises): {inputs.shape[1]}")
    print(f"Número de filas (altura): {inputs.shape[2]}")
    print(f"Número de columnas (ancho): {inputs.shape[3]}")
    
    # Salir del bucle después del primer lote, ya que todos los lotes tienen la misma forma
    break

Número de lotes (batch size): 32
Número de canales (RGB/escala de grises): 3
Número de filas (altura): 32
Número de columnas (ancho): 32


REALIZAMOS EL MODELO

In [10]:
from sklearn.metrics import accuracy_score
def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1, keepdims=True)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(SimpleCNN, self).__init__()
        # Capas convolucionales que procesan los tensores 2D de las imágenes
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=0)
        
        # Las capas lineales esperan un tensor aplanado. Aquí es donde se aplanan los datos.
        self.fc1 = nn.Linear(32 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # El tensor x se procesa sin ser aplanado
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        
        # Se aplana SOLO antes de pasar a la primera capa lineal (fc1)
        x = x.view(-1, 32 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:
def fit(model, dataloader, optimizer, epochs=120, log_each=10, early_stopping=0):
    criterion = nn.CrossEntropyLoss()
    l, acc, val_l, val_acc = [], [], [], []
    best_acc, step = 0, 0
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    for e in range(1, epochs + 1):
        _l, _acc = [], []
        model.train()
        for x_b, y_b in dataloader['train']:
            x_b, y_b = x_b.to(device), y_b.to(device)
            # Ya no necesitas aplanar x_b aquí
            
            y_pred = model(x_b) # El modelo se encarga de aplanar
            loss = criterion(y_pred, y_b)
            _l.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            y_probas = torch.argmax(softmax(y_pred), axis=1)
            _acc.append(accuracy_score(y_b.cpu().numpy(), y_probas.cpu().detach().numpy()))
        
        l.append(np.mean(_l))
        acc.append(np.mean(_acc))
        
        model.eval()
        _l, _acc = [], []
        with torch.no_grad():
            for x_b, y_b in dataloader['val']:
                x_b, y_b = x_b.to(device), y_b.to(device)
                # Ya no necesitas aplanar x_b aquí
                
                y_pred = model(x_b) # El modelo se encarga de aplanar
                loss = criterion(y_pred, y_b)
                _l.append(loss.item())
                y_probas = torch.argmax(softmax(y_pred), axis=1)
                _acc.append(accuracy_score(y_b.cpu().numpy(), y_probas.cpu().numpy()))
        
        val_l.append(np.mean(_l))
        val_acc.append(np.mean(_acc))
        
        if val_acc[-1] > best_acc:
            best_acc = val_acc[-1]
            torch.save(model.state_dict(), 'ckpt.pt')
            step = 0
            print(f"Mejor modelo guardado con acc {best_acc:.5f} en epoch {e}")
        step += 1
        if early_stopping and step > early_stopping:
            print(f"Entrenamiento detenido en epoch {e} por no mejorar en {early_stopping} epochs seguidas")
            break
        if not e % log_each:
            print(f"Epoch {e}/{epochs} loss {l[-1]:.5f} acc {acc[-1]:.5f} val_loss {val_l[-1]:.5f} val_acc {val_acc[-1]:.5f}")
    
    model.load_state_dict(torch.load('ckpt.pt'))
    return {'epoch': list(range(1, len(l)+1)), 'loss': l, 'acc': acc, 'val_loss': val_l, 'val_acc': val_acc}

Realizamos comparativas:

In [ ]:
num_classes = len(train_dataset.classes)
model = SimpleCNN(num_classes=num_classes)
optimizer = optim.SGD(model.parameters(), lr=0.01)
hist_sgd = fit(model, dataloader, optimizer)
print(hist_sgd)

Mejor modelo guardado con acc 0.40614 en epoch 1
